<a href="https://colab.research.google.com/github/MBilalQ/FinMMEval-Lab-2026/blob/main/FinnMMEval_Task2_Qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate datasets evaluate rouge_score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.5 MB/s eta 0:00:00


In [2]:
import torch
import evaluate
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
# --- 1. LOAD MODEL & TOKENIZER ---
# We use float16 because the Colab T4 GPU handles it faster than bfloat16
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [4]:
# 1. LOAD IN 4-BIT TO SAVE VRAM
print("Loading model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

Loading model in 4-bit...


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
# 2. LOAD DATASETS
print("Loading CLEF PolyFiQA datasets...")
ds_easy = load_dataset("TheFinAI/PolyFiQA-Easy", split="test")
ds_expert = load_dataset("TheFinAI/PolyFiQA-Expert", split="test")

Loading CLEF PolyFiQA datasets...


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/76 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/76 [00:00<?, ? examples/s]

In [7]:
# --- 3. SETUP METRICS ---
rouge = evaluate.load("rouge")

In [8]:
def evaluate_zero_shot(dataset, dataset_name):
    print(f"\nStarting evaluation for {dataset_name} ({len(dataset)} rows)...")
    predictions = []
    references = []

    for i, row in enumerate(dataset):
        messages = [
            {"role": "system", "content": "You are an expert multilingual financial AI. Answer the user's question accurately and concisely based strictly on the provided context."},
            {"role": "user", "content": f"{row['query']}\n\nQuestion: {row['question']}"}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # 3. TRUNCATION (THE OOM FIX)
        # We cap the maximum tokens to 5000 to prevent the T4 from crashing
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=10000
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False, # Strictly greedy decoding (removed temperature warning)
                pad_token_id=tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        pred_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        predictions.append(pred_text)
        references.append(row['answer'])

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(dataset)} rows...")

    results = rouge.compute(predictions=predictions, references=references)
    print(f"--- {dataset_name} Results ---")
    print(f"ROUGE-1: {results['rouge1']:.4f}")

    return results

In [9]:
easy_metrics = evaluate_zero_shot(ds_easy, "PolyFiQA-Easy")

expert_metrics = evaluate_zero_shot(ds_expert, "PolyFiQA-Expert")


Starting evaluation for PolyFiQA-Easy (76 rows)...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processed 10/76 rows...
Processed 20/76 rows...
Processed 30/76 rows...
Processed 40/76 rows...
Processed 50/76 rows...
Processed 60/76 rows...
Processed 70/76 rows...
--- PolyFiQA-Easy Results ---
ROUGE-1: 0.0732

Starting evaluation for PolyFiQA-Expert (76 rows)...
Processed 10/76 rows...
Processed 20/76 rows...
Processed 30/76 rows...
Processed 40/76 rows...
Processed 50/76 rows...
Processed 60/76 rows...
Processed 70/76 rows...
--- PolyFiQA-Expert Results ---
ROUGE-1: 0.0692
